# Week 11 — Train the residual diffusion model (11a)

**Training only.** This notebook loads the prebuilt `diffusion_windows_v2.parquet`
(from `11_00_build_v2.ipynb`) and trains residual-model experiments; evaluation
lives in `11b_evaluate.ipynb`. The split mirrors the empirical pair
(`11c_train_empirical` / `11d_evaluate_empirical`).

### What changed vs the old combined notebook
- **Model selection is on a distributional metric, not the denoising loss.**
  `train_experiment` checkpoints on **`val_emd`** (mean Wasserstein-1 between the
  generated and empirical latitude densities) — the denoising `val_loss` rises
  while sample quality improves, so it is a poor stopping signal. The best
  checkpoint is saved under `CKPT_DIR/select/<name>/`; the final-iterate weights
  still go to `ckpt_<name>.ckpt`.
- **Optional residual constraints** make residuals *proper*: opt in per
  experiment via cfg knobs `lambda_mass` (integrate-to-zero / mass conservation),
  `lambda_neg` (non-negativity of `par + residual`), and `lambda_band` (no mass
  outside the Spörer band). All default to 0 (unchanged behavior).
- The validation distributional suite (EMD, energy distance, CRPS, moment
  errors) is logged every `eval_every_n_epochs`.


In [ ]:
# Standard Week 10 setup: locate the repo, install if missing.

import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, glob, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.stats import norm as sp_norm
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import repeat

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, CSVLogger
import wandb


In [ ]:
# Bootstrap sys.path and locate artifacts.
import os, sys

# Several week directories ship a conditioned_infrastructure.py, but only the
# Week-10/11 copy defines find_week10_artifacts and the Extended* API — the
# Week-09 copy is an older stub. Force week_10 (the copy find_week10_artifacts
# itself resolves to, so no module eviction happens) to the FRONT of sys.path
# so the Week-09 stub can never shadow it, and add week_11 so eval mode can
# import its evaluation.py. Also drop any stale module a prior failed import
# may have cached as the Week-09 stub.
_week09_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_09"))
_week10_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_10"))
_week11_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_11"))
for _p in (_week09_dir, _week11_dir, _week10_dir):   # week_10 inserted last -> resolves first; week_09 trails so its conditioned_infrastructure stub never shadows but unconditioned_infrastructure stays importable
    if _p in sys.path:
        sys.path.remove(_p)
    sys.path.insert(0, _p)
sys.modules.pop("conditioned_infrastructure", None)

from conditioned_infrastructure import find_week10_artifacts
# parquet_v2 is built fresh in Part A every run; raw CSV is needed by the
# eval phase (Task 67 Part 1). Both are listed as required so a missing
# raw CSV fails loudly at setup time rather than at Task 67.
paths = find_week10_artifacts(extra_required=[
    "data/composite_sunspot_groups_peak_area.csv",
])
# find_week10_artifacts resolves conditioned_infrastructure to the week_10
# copy (its home, where the parquet + checkpoints live) and evicts any other
# cached copy. Repoint the *module* to the week_11 copy — that's where THIS
# notebook's API lives (E9–E11 + the cond_*_valid plumbing) — while keeping
# the week_10 artifact paths returned in `paths`.
if _week11_dir in sys.path:
    sys.path.remove(_week11_dir)
sys.path.insert(0, _week11_dir)
sys.modules.pop("conditioned_infrastructure", None)

from unconditioned_infrastructure import make_cosine_schedule
from conditioned_infrastructure import (
    ExtendedConditionalResidualDataset,
    ExtendedConditionalDiffusionLightning,
    train_experiment,
    load_trained_experiment,
    sample_conditional_extended,
    build_model,
    block_cond_concat,
    k_run_combined,
    discover_experiment_checkpoints,
)
from butterflAI_model import ButterflAIModel

import conditioned_infrastructure as _ci
print(f"using conditioned_infrastructure from: {_ci.__file__}")

_WEEK10_DIR = paths["week10_dir"]
PARQUET_V2  = os.path.join(_WEEK10_DIR, "diffusion_windows_v2.parquet")
CKPT_DIR    = _WEEK10_DIR

classical   = ButterflAIModel(paths["classical_weights"])

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)

# --- Device selection -------------------------------------------------------
# On a multi-GPU box, pick which CUDA device to train on by setting its index
# (0, 1, ...; matches `nvidia-smi`). Set CUDA_DEVICE = None to let Lightning
# auto-pick, or to fall back to CPU when no GPU is present. The chosen device
# is threaded into train_experiment() below so the Lightning Trainer pins to it.
CUDA_DEVICE = 3   # int GPU index, or None for auto / CPU

if torch.cuda.is_available() and CUDA_DEVICE is not None:
    n_gpu = torch.cuda.device_count()
    if CUDA_DEVICE >= n_gpu:
        raise ValueError(
            f"CUDA_DEVICE={CUDA_DEVICE} but only {n_gpu} CUDA device(s) visible "
            f"(valid indices 0..{n_gpu - 1}).")
    device = torch.device(f"cuda:{CUDA_DEVICE}")
    torch.cuda.set_device(device)
    print(f"device           : {device} ({torch.cuda.get_device_name(device)})")
else:
    device = torch.device("cpu")
    print(f"device           : {device}")

# Load the v1 parquet — we extend it but never modify it.
# Test split is reserved for the PI.
windows_v1 = pd.read_parquet(paths["parquet_v1"])
windows_v1 = windows_v1.loc[windows_v1["split"].isin(["train", "val"])].reset_index(drop=True)
print(f"v1 parquet (train+val only): {len(windows_v1)} rows")
print(f"  splits: {windows_v1['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles: {sorted(windows_v1['cycle'].unique())}")
print(f"v2 parquet target: {PARQUET_V2}")


In [ ]:
# Load the prebuilt v2 parquet (built by 11_00_build_v2.ipynb). This notebook
# never rebuilds it — if it's missing, run 11_00_build_v2 first.
assert os.path.isfile(PARQUET_V2), (
    f"{PARQUET_V2} not found — run 11_00_build_v2.ipynb first.")
windows_aug = pd.read_parquet(PARQUET_V2)
windows_aug = windows_aug.loc[windows_aug["split"].isin(["train", "val"])].reset_index(drop=True)
print(f"v2 parquet (train+val): {len(windows_aug)} rows, {len(windows_aug.columns)} cols")


---
## Part B — wandb setup and the experiment menu

### Task 64 — Per-student wandb project

Each student gets their **own** wandb project. Replace the placeholder
strings below with your handle. Every training run in this notebook
logs to that project with the experiment ID as the run name; you can
compare all your variants on a single dashboard.

If wandb is unavailable in your environment, training will fall back to
a local CSV logger automatically. The assertion guard runs in both
modes so eval mode also tells you if you forgot to personalize the
project name.


In [ ]:
# Task 64 — wandb identity.

WANDB_PROJECT = "butterflai-w11-amunoz-metrics"
WANDB_ENTITY  = None

### Task 65 — Design your own experiments

The Week 10 baseline (E0) reproduces the existing conditional
diffusion on the v2 parquet — no new knobs. Everything beyond it is
your call. Each variant you propose should change **one knob** from
the previous run and answer **one question**.

The knobs available are:

- **Cond groups** (`groups` / `consumed_keys`): `base`, plus any of
  `cyclehemi`, `opp`, `traj`.
- **Architecture** (`arch`): `concat` or `film`.
- **Classifier-free guidance**: `cond_dropout_p=0.1` at training time;
  10e sweeps the guidance weight at sampling.
- **Fourier lifting**: `fourier=True` lifts cond scalars via sin/cos.

The menu below escalates roughly by effort-per-insight. Pick what's
interesting, add a new entry to `EXPERIMENTS`, and progress one
variant per session.

**Level 1 — same cond, change the channel.**
Add one of the new cond groups (`cyclehemi`, `opp`, `traj`) to E0's
`consumed_keys`. *Does the diffusion's val NLL drop when given more
information, with the architecture held fixed?*

**Level 2 — same information, change the mechanism.**
Switch `arch` from `concat` to `film` while keeping `cond_base` only.
*Does the modulation mechanism alone close the gap with classical?*

**Level 3 — best information × best mechanism.**
Combine your best Level 1 cond set with FiLM. *Is the combined gain
additive, or did Level 2 already capture it?*

**Level 4 — guidance.**
Set `cond_dropout_p=0.1` and train. Sampling guidance is swept in 10e.
*Can sharpening the conditional density buy you margin over Level 3?*

**Level 5 — Fourier lifting.**
Set `fourier=True`. *Does sin/cos lifting of the cond scalars help the
network represent boundaries?*

You can go further — bump `hidden_dim` / `n_layers`, raise `K_LAGS`
back in Task 62, pair lagged opposite-hemisphere with trajectory, or
anything else you can defend. Different students should diverge here;
results pool in 10e.

In [ ]:
# Task 65 — experiment specs.

_BASE_TEMPLATE = {
    "arch":           "concat",
    "consumed_keys":  ["cond_base"],
    "groups":         ["base"],
    "hidden_dim":     128,
    "n_layers":       3,
    "fourier":        False,
    "cond_dropout_p": 0.0,
    "max_epochs":     20000,
    "lr":             1e-3,
    "batch_size":     64,
    "lambda_mass": 1.0,
    "lambda_neg": 1.0, 
    "eval_every_n_epochs": 200,
    "n_eval_windows": 512,
    "n_ensemble": 16
}

def _spec(**overrides):
    d = dict(_BASE_TEMPLATE); d.update(overrides); return d

EXPERIMENTS = {
    # Baseline — same 4-D cond, concat arch, on the v2 parquet.
    "E0metrics": _spec(),

    # Level 2 — FiLM architecture, base cond only.
    "E4metrics": _spec(arch="film"),

    # Fourier lifting on top of base conditioning.
    "E12metrics": _spec(fourier=True),

    # Fourier lifting with film on top of base conditioning.
    "E13metrics": _spec(arch="film",
                        fourier=True),

    # Fourier lifting on top of base conditioning.
    "E14metrics": _spec(cond_dropout_p=0.1),

    # Fourier lifting on top of base conditioning.
    "E15metrics": _spec(arch="film",
                cond_dropout_p=0.1),    

    # Fourier lifting on top of base conditioning.
    "E16metrics": _spec(fourier=True,
                cond_dropout_p=0.1),                  

    # Fourier lifting with film on top of base conditioning.
    "E17metrics": _spec(arch="film",
                        fourier=True,
                        cond_dropout_p=0.1),


    # Level 1 — one new cond group at a time, concat arch.
    "E1metrics": _spec(consumed_keys=["cond_base", "cond_cyclehemi"],
                groups=["base", "cyclehemi"]),
    "E2metrics": _spec(consumed_keys=["cond_base", "cond_opp"],
                groups=["base", "opp"]),
    "E3metrics": _spec(consumed_keys=["cond_base", "cond_traj"],
                groups=["base", "traj"]),

    # Level 3 — FiLM + best L1 cond group (opp).
    "E5metrics": _spec(arch="film",
                consumed_keys=["cond_base", "cond_opp"],
                groups=["base", "opp"]),

    # Level 4 — classifier-free guidance on top of L3.
    "E6metrics": _spec(arch="film",
                consumed_keys=["cond_base", "cond_opp"],
                groups=["base", "opp"],
                cond_dropout_p=0.1),

    # Level 5 — Fourier lifting on top of L3.
    "E7metrics": _spec(arch="film",
                consumed_keys=["cond_base", "cond_opp"],
                groups=["base", "opp"],
                fourier=True),

    # Fourier lifting on top of L3 and classifier free guidance.
    "E8metrics": _spec(arch="film",
                consumed_keys=["cond_base", "cond_opp"],
                groups=["base", "opp"],
                fourier=True,
                cond_dropout_p=0.1),


    # Fourier lifting on top of E3.
    "E9metrics": _spec(arch="film",
                consumed_keys=["cond_base", "cond_traj"],
                groups=["base", "traj"],
                fourier=True),

    # Fourier lifting on top of E3 and E2.
    "E10metrics": _spec(arch="film",
                consumed_keys=["cond_base", "cond_opp", "cond_traj"],
                groups=["base", "opp", "traj"],
                fourier=True),


    # Fourier lifting on top of E3 plus valida window info.
    "E11metrics": _spec(arch="film",
                consumed_keys=["cond_base", "cond_traj", "cond_traj_valid"],
                groups=["base", "traj"],
                fourier=True), 

                                                                                                                              
}

for name, cfg in EXPERIMENTS.items():
    print(f"{name}: arch={cfg['arch']:6s}  consumed={cfg['consumed_keys']}  "
          f"fourier={cfg['fourier']}  cond_dropout_p={cfg['cond_dropout_p']}")

### Selection & constraint knobs (optional)

Each experiment cfg may add (all optional, all default off):

| key | effect |
|---|---|
| `lambda_mass` | penalize net mass added by the residual (integrate-to-zero) |
| `lambda_neg`  | penalize negative reconstructed density `par + residual` |
| `lambda_band` | penalize mass outside the Spörer band (5–40°) |
| `eval_every_n_epochs` | cadence of the val distributional metrics + checkpoint (default 200) |
| `n_eval_windows`, `n_ensemble` | windows × samples for the in-loop val metrics |

Example: `_spec(lambda_mass=1.0, lambda_neg=1.0, eval_every_n_epochs=500)`.


In [ ]:
# Train. EDIT THIS LIST — one new experiment per session. The loop is
# idempotent: it skips any experiment whose ckpt_<name>.ckpt already exists.
# ENABLED_EXPERIMENTS = ["E0metrics", "E1metrics", "E2metrics", "E3metrics", "E4metrics", "E5metrics", "E6metrics", "E7metrics", "E8metrics", "E9metrics", "E10metrics", "E11metrics"]

ENABLED_EXPERIMENTS = ["E12metrics", "E13metrics", "E14metrics", "E15metrics", "E16metrics", "E17metrics"]

for _name in ENABLED_EXPERIMENTS:
    if _name not in EXPERIMENTS:
        raise KeyError(f"unknown experiment {_name!r}; defined: {list(EXPERIMENTS)}")
    train_experiment(
        name=_name, cfg=EXPERIMENTS[_name],
        windows_aug=windows_aug, ckpt_dir=CKPT_DIR,
        wandb_project=WANDB_PROJECT, wandb_entity=WANDB_ENTITY,
        alpha_np=alpha_np, sigma_np=sigma_np, T=T,
        bin_centers=BIN_CENTERS, bin_width=BIN_WIDTH,
        device=device,
    )
print("training complete; val_emd-selected checkpoints under", os.path.join(CKPT_DIR, "select"))
